# EXERKINEMAP — Exercise-Responsive Metabolomics & Proteomics QC

**Notebook:** `exerkine_motrpac_lcms_olink_multiomics_qc.ipynb`  
**Purpose:** reproducible analysis of processed MoTrPAC rat targeted/untargeted LC–MS/MS feature tables and human Olink protein NPX data, with shared QC, exploratory analysis, differential response testing, pathway mapping, and exportable result manifests.

This notebook follows the workflow principles in Hajnajafi & Iqbal (2025): sample/analytical QC, missingness assessment, normalization, metabolite annotation, statistical analysis, and pathway interpretation. It is designed for **processed quantification tables**, not vendor raw files. Vendor RAW, mzML, peak picking, alignment, identification, and quantification must be handled by validated assay-specific software before this notebook.

**MoTrPAC data:** use the official Data Hub and the assay-specific transfer/data dictionaries. Rat LC–MS/MS metabolomics and human Olink are distinct assay families; they must be processed separately and only integrated after sample metadata, biological context, units, and identifiers are harmonized. Do not merge rat and human observations as if they were the same cohort.

Sources: [Metabolomics workflow review](https://doi.org/10.1186/s12953-025-00241-8) · [MoTrPAC data deposition](https://motrpac-data.org/data-deposition) · [Metabolomics transfer protocol](https://www.motrpac-data.org/knowledge-center/data-ingestion/transfer-protocols/metabolomics) · [Olink/targeted proteomics transfer protocol](https://www.motrpac-data.org/knowledge-center/data-ingestion/transfer-protocols/targeted-proteomics)

> Research-use workflow. Statistical associations and pathway enrichment are not proof of mechanism or clinical biomarkers.

## 1. Analysis plan and input contract

Provide two tabular files per analysis: (1) a **wide feature matrix** with one row per sample and one column per analyte, plus a sample ID column; (2) a sample metadata table with the same unique sample IDs and experimental design columns. Feature values should be numeric abundances or Olink NPX values. Keep raw/vendor files immutable.

Recommended feature matrix formats: CSV/TSV/Parquet. For LC–MS/MS, feature IDs should be stable feature identifiers or curated metabolite IDs; preserve m/z, retention time, adduct, identification confidence and QC annotations in a separate feature annotation table where available. For Olink, retain assay/protein identifiers and NPX; NPX is a normalized log2-scale relative quantity, not an absolute concentration.

**Do not** apply generic LC–MS normalization to Olink NPX. The notebook branches normalization recommendations by modality.

In [ ]:
from pathlib import Path
import json, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf

SEED = 20261020
np.random.seed(SEED)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "molecular_profiles"
OUT_DIR = PROJECT_ROOT / "results" / "molecular_profiles"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version())
print("Working directory:", PROJECT_ROOT)
print("Output directory:", OUT_DIR)

## 2. Configure the analysis

Edit paths and column names to match the downloaded data dictionary. Set `MODALITY` to `lcms_targeted`, `lcms_untargeted`, or `olink`. The default uses placeholders and will stop clearly until real files are supplied. Avoid including direct identifiers in analysis tables.

In [ ]:
# ---- USER CONFIGURATION ----
MODALITY = "lcms_untargeted"  # lcms_targeted | lcms_untargeted | olink
MATRIX_PATH = DATA_DIR / "feature_matrix.csv"  # rows=samples, columns=features
METADATA_PATH = DATA_DIR / "sample_metadata.csv"
ANNOTATION_PATH = DATA_DIR / "feature_annotations.csv"  # optional; feature_id column expected

SAMPLE_ID = "sample_id"
GROUP_COL = "group"          # e.g. sedentary / exercise; use exact data dictionary vocabulary
SUBJECT_COL = "subject_id"   # repeated-measures/blocking ID; set None if independent samples
TIME_COL = "timepoint"       # optional; e.g. baseline, acute, recovery
BATCH_COL = "batch"           # optional analytical batch
SAMPLE_TYPE_COL = "sample_type" # optional: Sample, QC-Sample, blank, pooled-QC
REFERENCE_GROUP = None         # e.g. "sedentary"; None chooses first observed group

# Feature/sample QC thresholds are configurable, not universal biological constants.
MAX_FEATURE_MISSING = 0.40
MAX_SAMPLE_MISSING = 0.40
MIN_NONMISSING_PER_GROUP = 3
FDR_ALPHA = 0.05
TOP_N_PLOT = 25

if MODALITY not in {"lcms_targeted", "lcms_untargeted", "olink"}:
    raise ValueError("MODALITY must be lcms_targeted, lcms_untargeted, or olink")
if not MATRIX_PATH.exists() or not METADATA_PATH.exists():
    print("Input files not found yet. Place the processed matrix and metadata at the configured paths, then rerun this cell.")
else:
    print("Input files found.")

In [ ]:
def read_table(path):
    path = Path(path)
    if path.suffix.lower() in {'.tsv', '.txt'}:
        return pd.read_csv(path, sep='\t')
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)

def validate_inputs(matrix_path, metadata_path, sample_id):
    matrix = read_table(matrix_path)
    meta = read_table(metadata_path)
    for name, frame in [("matrix", matrix), ("metadata", meta)]:
        if sample_id not in frame.columns:
            raise ValueError(f"{name} is missing required sample ID column: {sample_id}")
        if frame[sample_id].isna().any() or frame[sample_id].duplicated().any():
            raise ValueError(f"{name} sample IDs must be nonmissing and unique")
    feature_cols = [c for c in matrix.columns if c != sample_id]
    matrix[feature_cols] = matrix[feature_cols].apply(pd.to_numeric, errors='coerce')
    missing_in_meta = sorted(set(matrix[sample_id]) - set(meta[sample_id]))
    missing_in_matrix = sorted(set(meta[sample_id]) - set(matrix[sample_id]))
    if missing_in_meta or missing_in_matrix:
        raise ValueError(f"Sample IDs differ. Matrix not in metadata: {missing_in_meta[:10]}; metadata not in matrix: {missing_in_matrix[:10]}")
    meta = meta.set_index(sample_id).loc[matrix[sample_id]].reset_index()
    return matrix, meta, feature_cols

if MATRIX_PATH.exists() and METADATA_PATH.exists():
    matrix, metadata, feature_cols = validate_inputs(MATRIX_PATH, METADATA_PATH, SAMPLE_ID)
    print(f"Samples: {len(matrix):,}; features: {len(feature_cols):,}; metadata columns: {list(metadata.columns)}")
else:
    matrix = metadata = None
    feature_cols = []

## 3. Metadata and assay integrity checks

Inspect sample counts, group balance, timepoint coverage, and batch allocation before looking at molecular outcomes. Unbalanced batch/group confounding cannot be repaired reliably by a normalization step alone. For repeated measures, ensure subject IDs and paired timepoints are correctly represented.

In [ ]:
if matrix is not None:
    display(metadata.head())
    print("Duplicate feature names:", pd.Index(feature_cols).duplicated().sum())
    print("Missing metadata by column:")
    display(metadata.isna().sum().sort_values(ascending=False).to_frame("n_missing").head(20))
    if GROUP_COL not in metadata.columns:
        raise ValueError(f"GROUP_COL={GROUP_COL!r} not found. Set it to a valid metadata column.")
    display(metadata[GROUP_COL].value_counts(dropna=False).rename_axis(GROUP_COL).to_frame("n_samples"))
    if BATCH_COL and BATCH_COL in metadata.columns:
        display(pd.crosstab(metadata[GROUP_COL], metadata[BATCH_COL], margins=True))
    if TIME_COL and TIME_COL in metadata.columns:
        display(pd.crosstab(metadata[GROUP_COL], metadata[TIME_COL], margins=True))

## 4. Sample and feature quality control

QC includes missingness, zero/constant features, total signal summaries, and distribution plots. For LC–MS/MS, review pooled-QC CV, blanks, internal standards, retention-time drift, carryover, and run-order effects using the assay's own QC annotations when available. For Olink, use NPX and assay/sample QC flags supplied by the platform; do not infer assay failure solely from a generic abundance threshold.

This notebook reports diagnostics; it does not automatically declare samples biologically invalid.

In [ ]:
if matrix is not None:
    X_raw = matrix.set_index(SAMPLE_ID)[feature_cols].copy()
    sample_missing = X_raw.isna().mean(axis=1)
    feature_missing = X_raw.isna().mean(axis=0)
    sample_qc = pd.DataFrame({"sample_id": X_raw.index, "fraction_missing": sample_missing.values,
                              "median_signal": X_raw.median(axis=1).values,
                              "n_detected_features": X_raw.notna().sum(axis=1).values})
    feature_qc = pd.DataFrame({"feature_id": X_raw.columns, "fraction_missing": feature_missing.values,
                               "n_detected_samples": X_raw.notna().sum(axis=0).values,
                               "variance": X_raw.var(axis=0, skipna=True).values})
    sample_qc.to_csv(OUT_DIR / "sample_qc_summary.csv", index=False)
    feature_qc.to_csv(OUT_DIR / "feature_qc_summary.csv", index=False)
    display(sample_qc.sort_values("fraction_missing", ascending=False).head())
    fig, ax = plt.subplots(figsize=(9, 4)); sample_missing.sort_values().plot.bar(ax=ax)
    ax.set(title="Per-sample missingness", xlabel="Sample", ylabel="Fraction missing"); plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(figsize=(7, 4)); feature_missing.hist(bins=40, ax=ax)
    ax.set(title="Per-feature missingness", xlabel="Fraction missing", ylabel="Number of features"); plt.tight_layout(); plt.show()
    print("QC summaries saved to", OUT_DIR)

In [ ]:
if matrix is not None:
    keep_samples = sample_missing <= MAX_SAMPLE_MISSING
    keep_features = (feature_missing <= MAX_FEATURE_MISSING) & (X_raw.nunique(dropna=True) > 1)
    X = X_raw.loc[keep_samples, keep_features].copy()
    meta = metadata.set_index(SAMPLE_ID).loc[X.index].copy()
    print(f"Retained {X.shape[0]}/{X_raw.shape[0]} samples and {X.shape[1]}/{X_raw.shape[1]} features using configured missingness/variance rules.")
    print("Important: confirm these thresholds against the assay protocol and QC controls before interpreting results.")

## 5. Modality-aware preprocessing

**LC–MS/MS:** abundance tables commonly require blank/contaminant review, pooled-QC filtering, run-order drift assessment, appropriate normalization (internal standards, dilution/biomass, or validated global methods), and a documented missing-value strategy. Avoid blind zero replacement; missingness may be left-censored. The code below uses log2 only for positive abundances and preserves missing values.

**Olink:** NPX is already log2-scaled and normalized by the assay workflow. Do not log-transform NPX again. Apply the vendor's QC flags and documented normalization strategy; handle LOD-related values according to the assay release and prespecified analysis plan.

The simple median-centering option below is a transparent exploratory fallback for LC–MS only—not a substitute for assay-specific validated normalization.

In [ ]:
if matrix is not None:
    X_num = X.astype(float).copy()
    if MODALITY.startswith("lcms"):
        # Invalid/nonpositive values become missing before log2. Confirm the instrument export's zero semantics first.
        X_num = X_num.where(X_num > 0)
        X_log = np.log2(X_num)
        # Optional exploratory median centering. Set False if a validated normalization was already applied.
        APPLY_MEDIAN_CENTERING = False
        if APPLY_MEDIAN_CENTERING:
            X_proc = X_log.subtract(X_log.median(axis=1, skipna=True), axis=0)
        else:
            X_proc = X_log.copy()
        preprocessing_note = "LC-MS: log2 positive abundance; normalization depends on validated assay protocol"
    else:
        X_proc = X_num.copy()  # Olink NPX is already on log2 scale
        preprocessing_note = "Olink: NPX retained on normalized log2 scale; no additional log transform"
    print(preprocessing_note)
    print("Processed matrix shape:", X_proc.shape)
    X_proc.to_csv(OUT_DIR / "processed_matrix_logscale.csv", index_label=SAMPLE_ID)

## 6. Exploratory analysis: distributions, correlations, PCA

PCA is descriptive. If batch, sex, tissue, training duration, timepoint, or species structure dominates the first components, investigate design and QC before interpreting exercise biology. For LC–MS data, use a documented imputation approach only when needed by the downstream method; the PCA display below uses feature-wise median imputation strictly for visualization.

In [ ]:
if matrix is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=X_proc, ax=ax, showfliers=False, color="lightsteelblue")
    ax.set(title=f"Feature distributions ({MODALITY})", xlabel="Sample", ylabel="Log-scale abundance / NPX")
    ax.tick_params(axis='x', rotation=90); plt.tight_layout(); plt.show()

    X_vis = X_proc.copy()
    X_vis = X_vis.loc[:, X_vis.notna().sum(axis=0) >= 3]
    X_vis = X_vis.fillna(X_vis.median(axis=0)).fillna(0)
    if X_vis.shape[0] >= 3 and X_vis.shape[1] >= 2:
        Z = StandardScaler().fit_transform(X_vis.T).T
        pca = PCA(n_components=2, random_state=SEED)
        coords = pca.fit_transform(Z)
        pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=X_vis.index).join(meta)
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue=GROUP_COL,
                        style=BATCH_COL if BATCH_COL in pca_df.columns else None, s=70, ax=ax)
        ax.set_title(f"PCA — {MODALITY}; median imputation used for visualization only")
        plt.tight_layout(); plt.show()
        print("Explained variance:", np.round(pca.explained_variance_ratio_, 3))
        pca_df.to_csv(OUT_DIR / "pca_coordinates.csv", index_label=SAMPLE_ID)

## 7. Differential response testing

This starter model performs a feature-wise two-group Welch test and BH-FDR correction. It is appropriate only for a simple, independent two-group contrast and is **not** the default for repeated measures or complex MoTrPAC designs. For longitudinal/repeated observations, use a prespecified mixed-effects model or paired method with subject-level blocking; for multi-factor designs, include the required covariates and interaction terms. Do not treat technical injections as independent biological replicates.

Choose a contrast explicitly. The output reports log2-scale difference in group means, Welch p-values, BH-adjusted p-values, and group sample counts.

In [ ]:
def bh_fdr(pvalues):
    p = np.asarray(pvalues, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0: return out
    pv = p[ok]; order = np.argsort(pv); ranked = pv[order]
    adj = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1].clip(0, 1)
    temp = np.empty_like(adj); temp[order] = adj; out[ok] = temp
    return out

if matrix is not None:
    levels = list(pd.Series(meta[GROUP_COL]).dropna().unique())
    print("Observed groups:", levels)
    if len(levels) == 2:
        group_a = REFERENCE_GROUP if REFERENCE_GROUP in levels else levels[0]
        group_b = next(g for g in levels if g != group_a)
        rows = []
        for feature in X_proc.columns:
            a = X_proc.loc[meta[GROUP_COL].to_numpy() == group_a, feature].dropna().to_numpy()
            b = X_proc.loc[meta[GROUP_COL].to_numpy() == group_b, feature].dropna().to_numpy()
            if min(len(a), len(b)) < MIN_NONMISSING_PER_GROUP:
                continue
            test = stats.ttest_ind(b, a, equal_var=False, nan_policy="omit")
            rows.append({"feature_id": feature, "group_reference": group_a, "group_comparison": group_b,
                         "n_reference": len(a), "n_comparison": len(b),
                         "mean_reference": np.mean(a), "mean_comparison": np.mean(b),
                         "difference_comparison_minus_reference": np.mean(b)-np.mean(a),
                         "p_value": test.pvalue})
        de = pd.DataFrame(rows)
        if not de.empty:
            de["fdr_bh"] = bh_fdr(de["p_value"])
            de = de.sort_values(["fdr_bh", "p_value"])
            de.to_csv(OUT_DIR / "differential_response_welch.csv", index=False)
            display(de.head(20))
        else:
            print("No features met the minimum nonmissing sample count.")
    else:
        print("Not running two-group Welch tests: select a two-level subset/contrast or implement the study-appropriate multivariable model.")

## 8. Visualize differential features

Volcano-style plots are exploratory. The effect size is the difference in mean log2 abundance/NPX; for LC–MS it approximates a log2 fold change only when the transformed abundance and normalization are appropriate. Multiple testing correction is essential.

In [ ]:
if matrix is not None and 'de' in globals() and not de.empty:
    plot_df = de.copy()
    plot_df["minus_log10_fdr"] = -np.log10(plot_df["fdr_bh"].clip(lower=1e-300))
    plot_df["significant"] = plot_df["fdr_bh"] < FDR_ALPHA
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=plot_df, x="difference_comparison_minus_reference", y="minus_log10_fdr",
                    hue="significant", palette={True:"crimson", False:"gray"}, ax=ax, s=35)
    ax.set(title="Differential response (exploratory)", xlabel=f"Mean difference: {group_b} − {group_a}", ylabel="−log10(BH FDR)")
    plt.tight_layout(); plt.show()
    de.head(TOP_N_PLOT).to_csv(OUT_DIR / "top_differential_features.csv", index=False)

## 9. Annotation and pathway mapping

For LC–MS/MS, pathway enrichment is defensible only for confidently annotated metabolites with an appropriate organism-specific compound identifier mapping (for example, HMDB, KEGG Compound, or a curated metabolite ID). Untargeted features without identification should remain feature IDs; do not assign pathway membership from m/z alone. For Olink, map protein identifiers to gene symbols/UniProt and use protein-focused gene-set analysis. Avoid treating all measured features as an unbiased genome-wide background.

This cell joins optional annotations to differential results and exports a reviewable candidate table. Pathway enrichment can be added with a chosen database and explicit universe after identifier curation.

In [ ]:
if matrix is not None and 'de' in globals() and not de.empty:
    if ANNOTATION_PATH.exists():
        annotation = read_table(ANNOTATION_PATH)
        if "feature_id" not in annotation.columns:
            raise ValueError("Annotation table must include feature_id to join safely")
        annotated_de = de.merge(annotation, on="feature_id", how="left", validate="many_to_one")
    else:
        annotated_de = de.copy()
    annotated_de.to_csv(OUT_DIR / "differential_features_annotated.csv", index=False)
    print("Annotated result table saved. Review identifier confidence and organism before pathway analysis.")
    display(annotated_de.head(15))

## 10. Cross-modal integration: LC–MS/MS + Olink

Integrate only after independent QC and normalization. First align sample IDs from the **same biological specimens**, or use a justified subject/timepoint key. Rat metabolomics and human Olink are different species/cohorts and must not be joined by row order. Across species, use explicit orthology mapping and keep species-specific estimates visible.

For matched samples, a basic analysis can correlate preselected, well-annotated metabolites and proteins. Correct for multiple testing and control for group/timepoint where possible. Correlation does not establish metabolite–protein causality.

In [ ]:
# Optional integration template. Configure a second, independently QC'd matrix and metadata.
INTEGRATE_MATCHED_MODALITY = False
SECOND_MATRIX_PATH = DATA_DIR / "second_modality_matrix.csv"
SECOND_SAMPLE_KEY = "sample_id"  # must refer to the same biological specimen or a documented mapping

if INTEGRATE_MATCHED_MODALITY:
    if not SECOND_MATRIX_PATH.exists():
        raise FileNotFoundError(SECOND_MATRIX_PATH)
    second = read_table(SECOND_MATRIX_PATH).set_index(SECOND_SAMPLE_KEY)
    primary = X_proc.copy()
    shared = primary.index.intersection(second.index)
    if len(shared) < 5:
        warnings.warn("Fewer than 5 matched specimens; cross-modal correlations are unstable.")
    paired_corr = primary.loc[shared].join(second.loc[shared], lsuffix="_mod1", rsuffix="_mod2").corr(method="spearman")
    paired_corr.to_csv(OUT_DIR / "matched_modalities_spearman_correlation.csv")
    print("Correlation matrix exported. Use only after confirming matched specimen identities and independent QC.")

## 11. Reproducibility manifest and outputs

Save input filenames, configuration, software versions, thresholds, and the date. For publication, also record data accessions, download timestamps/checksums, assay protocol versions, feature annotation provenance, excluded samples/features and reasons, model formulas, contrast definitions, and the exact environment lockfile. Do not put private participant identifiers into the manifest.

In [ ]:
manifest = {
    "notebook": "exerkine_motrpac_lcms_olink_multiomics_qc.ipynb",
    "analysis_date_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "modality": MODALITY,
    "matrix_path": str(MATRIX_PATH),
    "metadata_path": str(METADATA_PATH),
    "annotation_path": str(ANNOTATION_PATH) if ANNOTATION_PATH.exists() else None,
    "sample_id_column": SAMPLE_ID,
    "group_column": GROUP_COL,
    "subject_column": SUBJECT_COL,
    "time_column": TIME_COL,
    "batch_column": BATCH_COL,
    "max_feature_missing": MAX_FEATURE_MISSING,
    "max_sample_missing": MAX_SAMPLE_MISSING,
    "minimum_nonmissing_per_group": MIN_NONMISSING_PER_GROUP,
    "fdr_alpha": FDR_ALPHA,
    "random_seed": SEED,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "outputs_directory": str(OUT_DIR),
    "limitations": ["Processed matrices only; raw vendor files require assay-specific processing.",
                    "Two-group Welch test is not appropriate for repeated measures or complex designs.",
                    "PCA median imputation is visualization-only.",
                    "Pathway analysis requires curated IDs and an appropriate feature universe."]
}
with open(OUT_DIR / "analysis_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))
print("Results written to:", OUT_DIR.resolve())

## 12. MoTrPAC-specific handoff checklist

- [ ] Confirm species, study phase, tissue code, assay code, exercise condition, sex, training duration/timepoint, and biological replicate definition from the data dictionary.
- [ ] Download processed quantification and metadata from the official MoTrPAC Data Hub / public repository; preserve accession and source checksums.
- [ ] Keep targeted and untargeted LC–MS/MS workflows distinct; verify the platform-specific assay and data format.
- [ ] Review pooled-QC samples, blanks, internal standards, run order, batch, missingness, and annotation confidence before modeling.
- [ ] For human Olink, retain NPX, sample/assay QC flags, plate/batch information, and the documented normalization method.
- [ ] For longitudinal/repeated measures, fit subject-aware models; do not use the simple Welch-test cell as final inference.
- [ ] Predefine contrasts, covariates, exclusions, pathway database, feature universe, and FDR policy before interpreting results.
- [ ] Keep rat-to-human translation explicit with ortholog mapping and uncertainty; do not pool raw values across species or assay platforms.
- [ ] Archive environment, code commit, metadata, analysis manifest, and all exclusions for reproducibility.

### Key references
1. Hajnajafi K, Iqbal MA. *Mass-spectrometry based metabolomics: an overview of workflows, strategies, data analysis and applications.* Proteome Science. 2025;23:5. https://doi.org/10.1186/s12953-025-00241-8
2. MoTrPAC Data Hub — data deposition: https://motrpac-data.org/data-deposition
3. MoTrPAC Knowledge Center — metabolomics transfer protocol: https://www.motrpac-data.org/knowledge-center/data-ingestion/transfer-protocols/metabolomics
4. MoTrPAC Knowledge Center — Olink/targeted proteomics transfer protocol: https://www.motrpac-data.org/knowledge-center/data-ingestion/transfer-protocols/targeted-proteomics